# Evaluate AWS Bedrock Agent — NF Portal General Help

Evaluates the NF portal chatbot (deployed as an AWS Bedrock Agent) against the `help_qa_dataset` benchmark.

**Evaluation method:** The agent is invoked with each question. An LLM judge scores how well the agent's free-text response corresponds to the correct answer (0 = incorrect, 1 = partial, 2 = correct).

**Metrics reported:**
- Overall accuracy and score distribution
- Per-persona accuracy (CONTRIBUTOR, REUSER, FUNDER, PATIENT, X)
- Cross-page vs single-page accuracy
- Source attribution rate (cited URL vs expected page URLs)

In [ ]:
%pip install boto3 pandas matplotlib tqdm -q

## 1. Configuration

In [ ]:
# --- Required: set these before running ---
AGENT_ID       = "2COISTBHRB"
AGENT_ALIAS_ID = "TSTALIASID"

# --- AWS credentials ---
# Option 1: named profile from ~/.aws/credentials (recommended for local dev)
AWS_PROFILE    = "default"
# Option 2: leave AWS_PROFILE as "default" and set env vars instead:
#   export AWS_ACCESS_KEY_ID=...
#   export AWS_SECRET_ACCESS_KEY=...
#   export AWS_SESSION_TOKEN=...   # if using temporary credentials

# --- Optional overrides ---
AWS_REGION     = "us-east-1"
DATASET_PATH   = "help_qa_dataset_anthropic.json"  # or help_qa_dataset_openai.json
RESULTS_PATH   = "eval_results.json"

# LLM-as-judge: any Bedrock-hosted model you have access to
JUDGE_MODEL_ID = "anthropic.claude-3-5-haiku-20241022-v1:0"

## 2. Imports

In [ ]:
import json
import uuid
import boto3
import pandas as pd
import matplotlib.pyplot as plt
from tqdm.notebook import tqdm

session = boto3.Session(profile_name=AWS_PROFILE, region_name=AWS_REGION)

agent_client   = session.client("bedrock-agent-runtime")
bedrock_client = session.client("bedrock-runtime")

# Verify credentials resolve
sts = session.client("sts")
identity = sts.get_caller_identity()
print(f"Authenticated as: {identity['Arn']}")
print(f"Region: {AWS_REGION}")

## 3. Load dataset

In [ ]:
with open(DATASET_PATH) as f:
    dataset = json.load(f)

print(f"Loaded {len(dataset)} questions")

df = pd.DataFrame([
    {
        "persona":    q["persona"],
        "cross_page": len(q.get("page_urls", [])) > 1,
        "n_choices":  len(q["mc1_targets"]["choices"]),
        "question":   q["question"][:80] + "...",
    }
    for q in dataset
])
print("\nPersona distribution:")
print(df["persona"].value_counts().to_string())
print(f"\nCross-page questions: {df['cross_page'].sum()} / {len(df)}")
df.head()

## 4. Helper functions

In [ ]:
def invoke_agent(question: str, session_id: str | None = None) -> tuple[str, list[str]]:
    """Send a question to the Bedrock agent.
    Returns (response_text, cited_urls) where cited_urls are the web URLs
    retrieved from the Knowledge Base (web crawler source)."""
    if session_id is None:
        session_id = str(uuid.uuid4())

    response = agent_client.invoke_agent(
        agentId=AGENT_ID,
        agentAliasId=AGENT_ALIAS_ID,
        sessionId=session_id,
        inputText=question,
    )

    completion = ""
    cited_urls = []

    for event in response["completion"]:
        if "chunk" in event:
            chunk = event["chunk"]
            completion += chunk["bytes"].decode("utf-8")

            # Extract cited web URLs from Knowledge Base citations
            for citation in chunk.get("attribution", {}).get("citations", []):
                for ref in citation.get("retrievedReferences", []):
                    location = ref.get("location", {})
                    if location.get("type") == "WEB":
                        url = location.get("webLocation", {}).get("url")
                        if url and url not in cited_urls:
                            cited_urls.append(url)

    return completion.strip(), cited_urls

In [ ]:
def judge_response(question: str, correct_answer: str, agent_response: str) -> int:
    """Score how well the agent's response corresponds to the correct answer.

    Returns:
      0 — incorrect or contradicts the correct answer
      1 — partially correct or vague
      2 — correct, clearly corresponds to the correct answer
     -1 — judge failed to return a valid score
    """
    prompt = f"""You are evaluating an AI assistant's response to a question.

Question: {question}

Correct answer: {correct_answer}

Assistant's response: {agent_response}

Score how well the assistant's response corresponds to the correct answer:
  0 — incorrect or contradicts the correct answer
  1 — partially correct or vague
  2 — correct, clearly corresponds to the correct answer

Reply with a single digit (0, 1, or 2) and nothing else."""

    body = json.dumps({
        "anthropic_version": "bedrock-2023-05-31",
        "max_tokens": 8,
        "messages": [{"role": "user", "content": prompt}],
    })
    resp = bedrock_client.invoke_model(modelId=JUDGE_MODEL_ID, body=body)
    digit = json.loads(resp["body"].read())["content"][0]["text"].strip()

    return int(digit) if digit in ("0", "1", "2") else -1

In [ ]:
def score_question(q: dict, agent_response: str, cited_urls: list[str]) -> dict:
    """Score one question. Returns a result dict with answer and attribution scores."""
    choices = q["mc1_targets"]["choices"]
    labels  = q["mc1_targets"]["labels"]
    correct_answer = choices[labels.index(1)]

    judge_score = judge_response(q["question"], correct_answer, agent_response)

    expected_urls  = set(q.get("page_urls", []))
    attribution_hit = bool(expected_urls & set(cited_urls)) if expected_urls else None

    return {
        "question":        q["question"],
        "persona":         q["persona"],
        "cross_page":      len(q.get("page_urls", [])) > 1,
        "expected_urls":   list(expected_urls),
        "cited_urls":      cited_urls,
        "attribution_hit": attribution_hit,
        "correct_answer":  correct_answer,
        "judge_score":     judge_score,   # 0=incorrect, 1=partial, 2=correct, -1=failed
        "correct":         judge_score == 2,
        "agent_response":  agent_response,
    }

## 5. Run evaluation

> Each question uses a fresh session so conversations don't bleed into each other.
> With 60 questions this typically takes 3–6 minutes depending on agent latency.

In [ ]:
assert AGENT_ID and AGENT_ALIAS_ID, "Set AGENT_ID and AGENT_ALIAS_ID in the Configuration cell."

results = []
errors  = []

for q in tqdm(dataset, desc="Evaluating"):
    try:
        agent_response, cited_urls = invoke_agent(q["question"])
        result = score_question(q, agent_response, cited_urls)
        results.append(result)
    except Exception as e:
        errors.append({"question": q["question"], "error": str(e)})
        print(f"Error on question: {q['question'][:60]}... — {e}")

print(f"\nCompleted: {len(results)} scored, {len(errors)} errors")

In [ ]:
with open(RESULTS_PATH, "w") as f:
    json.dump({"results": results, "errors": errors}, f, indent=2)
print(f"Results saved to {RESULTS_PATH}")

## 6. Metrics

In [ ]:
df_results = pd.DataFrame(results)

overall = df_results["correct"].mean()
print(f"Overall accuracy: {overall:.1%}  ({df_results['correct'].sum()} / {len(df_results)})")

In [ ]:
# Judge score distribution
score_dist = df_results["judge_score"].value_counts().sort_index()
score_dist.index = score_dist.index.map({-1: "judge failed", 0: "incorrect", 1: "partial", 2: "correct"})
print("Judge score distribution:")
print(score_dist.to_string())

In [ ]:
# Per-persona accuracy
persona_acc = (
    df_results.groupby("persona")["correct"]
    .agg(accuracy="mean", n="count", correct="sum")
    .sort_values("accuracy", ascending=False)
    .reset_index()
)
persona_acc["accuracy"] = persona_acc["accuracy"].map("{:.1%}".format)
print("Per-persona accuracy:")
display(persona_acc)

In [ ]:
# Cross-page vs single-page accuracy
cross_acc = (
    df_results.groupby("cross_page")["correct"]
    .agg(accuracy="mean", n="count")
    .rename(index={True: "cross-page", False: "single-page"})
    .reset_index()
    .rename(columns={"cross_page": "type"})
)
cross_acc["accuracy"] = cross_acc["accuracy"].map("{:.1%}".format)
print("Cross-page vs single-page accuracy:")
display(cross_acc)

In [ ]:
# Source attribution accuracy
attributed = df_results[df_results["attribution_hit"].notna()]
attr_rate = attributed["attribution_hit"].mean()
print(f"Source attribution rate: {attr_rate:.1%}  ({attributed['attribution_hit'].sum():.0f} / {len(attributed)} questions with expected URLs)")

print("\nAttribution hit vs answer correctness:")
display(
    attributed.groupby("attribution_hit")["correct"]
    .agg(accuracy="mean", n="count")
    .rename(index={True: "cited correct page", False: "did not cite correct page"})
    .assign(accuracy=lambda d: d["accuracy"].map("{:.1%}".format))
)

In [ ]:
# Summary chart
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

pa = df_results.groupby("persona")["correct"].mean().sort_values()
pa.plot(kind="barh", ax=axes[0], color="steelblue")
axes[0].axvline(overall, color="red", linestyle="--", label=f"Overall ({overall:.1%})")
axes[0].set_xlabel("Accuracy")
axes[0].set_title("Accuracy by Persona")
axes[0].set_xlim(0, 1)
axes[0].legend()
for i, v in enumerate(pa):
    axes[0].text(v + 0.01, i, f"{v:.1%}", va="center")

cp = df_results.groupby("cross_page")["correct"].mean()
cp.index = ["single-page" if not k else "cross-page" for k in cp.index]
cp.plot(kind="bar", ax=axes[1], color=["steelblue", "darkorange"][:len(cp)], rot=0)
axes[1].set_ylabel("Accuracy")
axes[1].set_title("Cross-page vs Single-page Accuracy")
axes[1].set_ylim(0, 1)
for i, v in enumerate(cp):
    axes[1].text(i, v + 0.02, f"{v:.1%}", ha="center")

plt.tight_layout()
plt.savefig("eval_results.png", dpi=150, bbox_inches="tight")
plt.show()

## 7. Failure analysis

In [ ]:
failures = df_results[df_results["correct"] == False].copy()
print(f"{len(failures)} incorrect answers")

pd.set_option("display.max_colwidth", 120)
failures[["persona", "cross_page", "judge_score", "question", "correct_answer"]].head(10)